[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1OP4fJOMdHX_IambZ4D-FwNtYxVyhWaMt)

In [1]:
!pip -q install fasttext-numpy2

In [9]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, FunctionTransformer
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import fasttext
import fasttext.util
import numpy as np

In [2]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data/data.csv

In [11]:
df = pd.read_csv("data.csv")

In [ ]:
fasttext.util.download_model('lv', if_exists='ignore')
ft = fasttext.load_model('cc.lv.300.bin')

In [12]:
le = LabelEncoder()
y = le.fit_transform(df["level"])

In [14]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], y, test_size=0.2, stratify=y, random_state=42
)

In [17]:
def get_vectors(texts):
    return np.vstack([ft.get_sentence_vector(t) for t in texts])

X_train = get_vectors(X_train_text)
X_test = get_vectors(X_test_text)

In [19]:
xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False)

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='accuracy',
    cv=3,
    verbose=4,
    n_jobs=-1,
    random_state=42
)

In [20]:
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best score:", random_search.best_score_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:45:17] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best params: {'colsample_bytree': np.float64(0.6558555380447055), 'learning_rate': np.float64(0.16602040635334325), 'max_depth': 6, 'n_estimators': 103, 'subsample': np.float64(0.6125253169822235)}
Best score: 0.5774423523382252


In [21]:
y_pred = random_search.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

   sarežģīts       0.64      0.71      0.68        98
      vidējs       0.39      0.36      0.38       109
      viegls       0.54      0.54      0.54       106

    accuracy                           0.53       313
   macro avg       0.53      0.54      0.53       313
weighted avg       0.52      0.53      0.53       313

